In [36]:
import pandas as pd

df = pd.read_csv('./drive/MyDrive/Colab Notebooks/100_Unique_QA_Dataset.csv')

df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [37]:
# tokenize
def tokenize(text):
  text = text.lower()
  text = text.replace('.', '')
  text = text.replace(',', '')
  text = text.replace('?', '')
  text = text.replace('!', '')
  text = text.replace("'", "")
  return text.split()

In [38]:
tokenize('What is the capital of France?')

['what', 'is', 'the', 'capital', 'of', 'france']

In [39]:
# vocab
vocab = {'<UNK>':0}

In [40]:
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])

  merged_tokens = tokenized_question + tokenized_answer

  for token in merged_tokens:

    if token not in vocab:
      vocab[token] = len(vocab)

In [41]:
df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [42]:
# convert words to numerical indices
def text_to_indices(text, vocab):

  indexed_text = []

  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [43]:
text_to_indices("What is campusX", vocab)

[1, 2, 0]

In [44]:
# vocab

In [45]:
import torch
from torch.utils.data import Dataset, DataLoader

In [46]:
class QADataset(Dataset):

  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):
    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [47]:
dataset = QADataset(df, vocab)

In [48]:
print(dataset[0])
print(dataset[10])

(tensor([1, 2, 3, 4, 5, 6]), tensor([7]))
(tensor([ 1,  2,  3,  4,  5, 53]), tensor([54]))


In [49]:
dataloader = DataLoader(dataset, batch_size=1 , shuffle=True)

In [50]:
for question, answer in dataloader:
  print(question, answer)

tensor([[ 1,  2,  3,  4,  5, 73]]) tensor([[74]])
tensor([[ 10, 308,   3, 309, 310]]) tensor([[311]])
tensor([[ 78,  79, 195,  81,  19,   3, 196, 197, 198]]) tensor([[199]])
tensor([[10, 75, 76]]) tensor([[77]])
tensor([[  1,   2,   3,   4,   5, 109]]) tensor([[317]])
tensor([[42, 86, 87, 88, 89, 39, 90]]) tensor([[91]])
tensor([[ 1,  2,  3, 33, 34,  5, 35]]) tensor([[36]])
tensor([[ 42, 167,   2,   3,  17, 168, 169]]) tensor([[170]])
tensor([[  1,   2,   3, 221,   5, 222, 223, 224]]) tensor([[225]])
tensor([[ 42,   2,   3, 274, 211, 275]]) tensor([[276]])
tensor([[ 42, 290, 291, 118, 292, 158, 293, 294]]) tensor([[295]])
tensor([[ 1,  2,  3, 69,  5, 53]]) tensor([[260]])
tensor([[ 78,  79, 261, 151,  14, 262, 153]]) tensor([[36]])
tensor([[  1,   2,   3,  37,  38,  39, 161]]) tensor([[162]])
tensor([[ 42, 263, 264,  14, 265, 266, 158, 267]]) tensor([[268]])
tensor([[ 42, 174,   2,  62,  39, 175, 176,  12, 177, 178]]) tensor([[179]])
tensor([[ 10,  11, 189, 158, 190]]) tensor([[191]])


In [51]:
import torch.nn as nn

In [52]:
class SimpleRNN(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50, 64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output

In [53]:
dataset[0][0]

tensor([1, 2, 3, 4, 5, 6])

In [54]:
x = nn.Embedding(324, embedding_dim=50)

In [55]:
a = x(dataset[0][0])

In [56]:
y = nn.RNN(50, 64)

In [57]:
# y(a)

In [58]:
y(a)[0].shape     # hidden states of RNN

torch.Size([6, 64])

In [59]:
y(a)[1].shape     # final output state of RNN

torch.Size([1, 64])

In [60]:
b = y(a)[1]

In [61]:
z = nn.Linear(64, 324)

In [62]:
# z(b)

In [63]:
z(b).shape

torch.Size([1, 324])

In [64]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print('Shape of a: ', a.shape)
b = x(a)
print('Shape of b: ', b.shape)
c, d = y(b)
print('Shape of c: ', c.shape)
print('Shape of d: ', d.shape)

e = z(d.squeeze(0))
print('Shape of e: ', e.shape)

Shape of a:  torch.Size([1, 6])
Shape of b:  torch.Size([1, 6, 50])
Shape of c:  torch.Size([1, 6, 64])
Shape of d:  torch.Size([1, 1, 64])
Shape of e:  torch.Size([1, 324])


In [65]:
learning_rate = 0.001
epochs = 20

In [66]:
model = SimpleRNN(len(vocab))

In [67]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [68]:
# training loop

for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # loss
    loss = criterion(output, answer[0])

    # gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f'Epoch: {epoch+1}, Loss: {total_loss:.4f}')

Epoch: 1, Loss: 522.7596
Epoch: 2, Loss: 457.2913
Epoch: 3, Loss: 381.7461
Epoch: 4, Loss: 314.6523
Epoch: 5, Loss: 259.2021
Epoch: 6, Loss: 209.3235
Epoch: 7, Loss: 165.7538
Epoch: 8, Loss: 127.9635
Epoch: 9, Loss: 98.0693
Epoch: 10, Loss: 74.8501
Epoch: 11, Loss: 57.8866
Epoch: 12, Loss: 45.6317
Epoch: 13, Loss: 35.7276
Epoch: 14, Loss: 29.1339
Epoch: 15, Loss: 23.4654
Epoch: 16, Loss: 19.5302
Epoch: 17, Loss: 16.3913
Epoch: 18, Loss: 13.8781
Epoch: 19, Loss: 11.9484
Epoch: 20, Loss: 10.3817


In [82]:
from numpy._core import numeric
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probability
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max probability
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [87]:
predict(model, 'What is the capital of france')

tensor([0.8600], grad_fn=<MaxBackward0>) tensor([7])


In [89]:
# vocab
# print(list(vocab.keys())[7])

paris
